In [ ]:
with open('results/participant_parcel_enrichment_results.pkl', 'rb') as f:
    enrichment_results = pickle.load(f)

enrichment_variables = {
    'Modeling Performance': None,  # filled from participant_means
    'Age': 'age',
    'Years of Experience': 'years_experience',
}

for m in best_models:
    layers = sorted(participant_means[m].keys())

    # average performance across layers for each participant
    all_perf_pids = set(pid for layer_num in layers for pid in participant_means[m][layer_num])
    layer_averaged_perf = {}
    for pid in all_perf_pids:
        vals = [participant_means[m][layer_num][pid] for layer_num in layers if pid in participant_means[m][layer_num]]
        layer_averaged_perf[pid] = float(np.mean(vals))

    # average enriched parcel counts across layers for each participant
    enrich_by_pid = {}
    for layer_num in layers:
        layer_key = f'layer_{layer_num}'
        if layer_key not in enrichment_results[m]:
            continue
        layer_enrich = enrichment_results[m][layer_key]
        for pid_str, val in layer_enrich.items():
            pid = int(pid_str)
            enrich_by_pid.setdefault(pid, []).append(val['num_sig_parcels'])

    layer_averaged_enrich = {pid: float(np.mean(vals)) for pid, vals in enrich_by_pid.items()}

    common_ids = [
        pid for pid in layer_averaged_enrich
        if pid in layer_averaged_perf and pid in demo_data.index
    ]
    enriched_vals = np.array([layer_averaged_enrich[pid] for pid in common_ids])

    fig, axes = plt.subplots(1, len(enrichment_variables), figsize=(14, 4))
    fig.suptitle(f"{m} | avg across {len(layers)} layers — enriched parcels vs.", fontsize=12, fontweight='bold')

    for ax, (var_label, col) in zip(axes, enrichment_variables.items()):
        if col is None:
            other_vals = np.array([layer_averaged_perf[pid] for pid in common_ids])
            ax.set_xlabel('Mean z-scored correlation')
        else:
            other_vals = demo_data.loc[common_ids, col].values.astype(float)
            ax.set_xlabel(var_label)

        mask = ~np.isnan(other_vals)
        x, y = other_vals[mask], enriched_vals[mask]

        r, p = stats.pearsonr(x, y)

        ax.scatter(x, y, color='steelblue', edgecolors='white', linewidths=0.5, s=60)
        m_fit, b_fit = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, m_fit * x_line + b_fit, color='tomato', linewidth=1.5)

        ax.set_ylabel('Num enriched parcels' if ax == axes[0] else '')
        ax.set_title(f"r = {r:.3f}, p = {p:.3f}")

    plt.tight_layout()
    # plt.savefig(f"results/enrichment_corr_{m}_avg_layers.png", dpi=150, bbox_inches='tight')
    plt.show()